# STCANet on Market-1501 — Person Re-Identification
### (Spatio-Temporal Channel Attention Network — image-ReID version)

এই notebook টা paper অনুযায়ী **Market-1501** dataset-এ person re-identification মডেল implement করে।

**কী কী থাকছে:**
1. Setup ও dataset path যাচাই
2. Dataset loader (Market-1501 ফরম্যাট পড়া)
3. Model: ResNet50 backbone + **SAM** (Spatial Attention) + **CAM** (Channel Attention) + STCA block
4. Loss function (ID loss + Triplet loss)
5. Evaluation (mAP + CMC)
6. Re-ranking (k-reciprocal)
7. Training loop
8. Final evaluation




## ধাপ ১ — Setup: GPU check + Dataset path বের করা

প্রথমে GPU অন আছে কিনা চেক করি, আর dataset ঠিক কোন path-এ আছে সেটা বের করি।

⚠️ **এই cell রান করার আগে:** Notebook-এর ডানপাশে **"+ Add Input"** থেকে Market-1501 dataset add করে নাও (search: `market 1501`)। তারপর Settings → Accelerator → **GPU T4 x2 / P100** সিলেক্ট করো।


In [18]:
import torch, os

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

print("\n--- /kaggle/input contents ---")
print(os.listdir("/kaggle/input"))


CUDA available: True
GPU: Tesla T4

--- /kaggle/input contents ---
['datasets']


In [19]:
# উপরের output দেখে dataset ফোল্ডারের নাম বসাও, তারপর ভেতরের structure দেখো
DATASET_FOLDER = os.listdir("/kaggle/input")[0]   # প্রয়োজনে ম্যানুয়ালি বসাও, যেমন: "market-1501"
print("Using:", DATASET_FOLDER)
print(os.listdir(f"/kaggle/input/{DATASET_FOLDER}"))


Using: datasets
['jiniyatanjina']


উপরের output-এ `bounding_box_train`, `query`, `bounding_box_test` ফোল্ডার তিনটা কোন লেভেলে আছে দেখে নাও —
কখনো কখনো একটা বাড়তি সাব-ফোল্ডার (যেমন `Market-1501-v15.09.15`) থাকে। সেই অনুযায়ী নিচের `DATA_ROOT` ঠিক করে দাও।


In [20]:
DATA_ROOT = "/kaggle/input/datasets/jiniyatanjina/market1501/Market-1501-v15.09.15"

TRAIN_DIR = "bounding_box_train"
QUERY_DIR = "query"
GALLERY_DIR = "bounding_box_test"

# sanity check
for d in [TRAIN_DIR, QUERY_DIR, GALLERY_DIR]:
    p = os.path.join(DATA_ROOT, d)
    print(p, "->", "EXISTS" if os.path.isdir(p) else "MISSING! path thik koro")

/kaggle/input/datasets/jiniyatanjina/market1501/Market-1501-v15.09.15/bounding_box_train -> EXISTS
/kaggle/input/datasets/jiniyatanjina/market1501/Market-1501-v15.09.15/query -> EXISTS
/kaggle/input/datasets/jiniyatanjina/market1501/Market-1501-v15.09.15/bounding_box_test -> EXISTS


## ধাপ ২ — Config

সব hyperparameter এক জায়গায়। Paper-এর Section 4.4 থেকে নেওয়া মান (`lr=0.00035`, `epoch=60`, `batch=64`, `triplet margin=0.3`)।

⚠️ **প্রথম রানে `num_epochs` অনেক কমিয়ে (যেমন 3-5) দিয়ে টেস্ট করো — সব ঠিক আছে কিনা দেখে তারপর বাড়াও।** নাহলে GPU সময় নষ্ট হতে পারে যদি মাঝপথে কোনো bug থাকে।


In [21]:
class Config:
    img_height = 256
    img_width = 128

    num_parts = 4          # head, upper body, lower body, shoes (horizontal stripes)

    batch_size_train = 64
    batch_size_test = 32
    num_instances = 4       # per identity per batch (PK sampler, triplet loss-এর জন্য দরকার)
    lr = 0.00035
    lr_decay_epochs = 20
    lr_decay_factor = 0.1
    num_epochs = 50         # <-- প্রথমে ছোট রাখো (sanity check), পরে 30-60 করো
    triplet_margin = 0.3
    beta1 = 1.0              # triplet loss-এর weight

    num_workers = 2
    checkpoint_dir = "/kaggle/working/checkpoints"
    log_interval = 50

cfg = Config()
os.makedirs(cfg.checkpoint_dir, exist_ok=True)


## ধাপ ৩ — Dataset Loader

Market-1501-এর ফাইলনেম ফরম্যাট: `0001_c1s1_001051_00.jpg`
- `0001` → person ID
- `c1` → camera ID
- বাকিটা frame/box info (আমাদের দরকার নেই)

`pid == -1` মানে junk image — এগুলো train set থেকে বাদ দিতে হয়।

**PKSampler** কেন লাগে? Triplet loss কাজ করতে হলে প্রতিটা batch-এ একই identity-র একাধিক image থাকা দরকার (positive pair বানাতে)। সাধারণ random shuffle এটা guarantee করে না, তাই আমরা "P identities × K images each" — এভাবে batch বানাই।


In [23]:
import re
import random
from collections import defaultdict
from torch.utils.data import Dataset, Sampler
from PIL import Image
import torchvision.transforms as T


def process_dir(dir_path, relabel=False):
    """Market-1501 ফোল্ডার স্ক্যান করে (img_path, pid, camid) লিস্ট বানায়."""
    img_paths = [os.path.join(dir_path, f) for f in os.listdir(dir_path) if f.endswith(".jpg")]
    pattern = re.compile(r"([-\d]+)_c(\d)")
    pid_container = set()

    for img_path in img_paths:
        m = pattern.search(os.path.basename(img_path))
        if m is None:
            continue
        pid = int(m.group(1))
        if pid == -1:
            continue
        pid_container.add(pid)

    pid2label = {pid: label for label, pid in enumerate(sorted(pid_container))}
    data = []
    for img_path in img_paths:
        m = pattern.search(os.path.basename(img_path))
        if m is None:
            continue
        pid, camid = int(m.group(1)), int(m.group(2))
        if pid == -1:
            continue
        camid -= 1
        if relabel:
            pid = pid2label[pid]
        data.append((img_path, pid, camid))
    return data, len(pid_container)


class Market1501(Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, pid, camid = self.data[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, pid, camid, img_path


def get_transforms(cfg, train=True):
    if train:
        return T.Compose([
            T.Resize((cfg.img_height, cfg.img_width)),
            T.RandomHorizontalFlip(p=0.5),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            T.RandomErasing(p=0.5),
        ])
    return T.Compose([
        T.Resize((cfg.img_height, cfg.img_width)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])


class PKSampler(Sampler):
    """P identities x K instances per batch (triplet loss-এর জন্য দরকার)."""
    def __init__(self, data, batch_size, num_instances=4):
        self.num_instances = num_instances
        self.num_pids_per_batch = batch_size // num_instances
        self.index_dic = defaultdict(list)
        for index, (_, pid, _) in enumerate(data):
            self.index_dic[pid].append(index)
        self.pids = list(self.index_dic.keys())
        self.length = len(self.pids) * num_instances

    def __iter__(self):
        batch_idxs_dict = defaultdict(list)
        for pid in self.pids:
            idxs = self.index_dic[pid].copy()
            if len(idxs) < self.num_instances:
                idxs = idxs * (self.num_instances // len(idxs) + 1)
            random.shuffle(idxs)
            batch_idxs_dict[pid] = idxs[: (len(idxs) // self.num_instances) * self.num_instances]

        avai_pids = list(self.pids)
        final_idxs = []
        while len(avai_pids) >= self.num_pids_per_batch:
            selected_pids = random.sample(avai_pids, self.num_pids_per_batch)
            for pid in selected_pids:
                idxs = batch_idxs_dict[pid][: self.num_instances]
                batch_idxs_dict[pid] = batch_idxs_dict[pid][self.num_instances:]
                final_idxs.extend(idxs)
                if len(batch_idxs_dict[pid]) < self.num_instances:
                    avai_pids.remove(pid)
        return iter(final_idxs)

    def __len__(self):
        return self.length

print("Dataset classes defined OK")


Dataset classes defined OK


এবার train/query/gallery ফোল্ডার পড়ে `DataLoader` বানাই।

In [24]:
from torch.utils.data import DataLoader

train_data, num_train_pids = process_dir(os.path.join(DATA_ROOT, TRAIN_DIR), relabel=True)
query_data, _ = process_dir(os.path.join(DATA_ROOT, QUERY_DIR), relabel=False)
gallery_data, _ = process_dir(os.path.join(DATA_ROOT, GALLERY_DIR), relabel=False)

print(f"Train IDs: {num_train_pids} | Train imgs: {len(train_data)} | "
      f"Query: {len(query_data)} | Gallery: {len(gallery_data)}")

train_set = Market1501(train_data, transform=get_transforms(cfg, train=True))
query_set = Market1501(query_data, transform=get_transforms(cfg, train=False))
gallery_set = Market1501(gallery_data, transform=get_transforms(cfg, train=False))

train_sampler = PKSampler(train_data, cfg.batch_size_train, cfg.num_instances)
train_loader = DataLoader(train_set, batch_size=cfg.batch_size_train,
                           sampler=train_sampler, num_workers=cfg.num_workers, drop_last=True)
query_loader = DataLoader(query_set, batch_size=cfg.batch_size_test, shuffle=False,
                           num_workers=cfg.num_workers)
gallery_loader = DataLoader(gallery_set, batch_size=cfg.batch_size_test, shuffle=False,
                             num_workers=cfg.num_workers)

# quick check -- এক ব্যাচ টেনে দেখি shape ঠিক আছে কিনা
imgs, pids, camids, paths = next(iter(train_loader))
print("Batch shape:", imgs.shape, "| pids sample:", pids[:8].tolist())


Train IDs: 751 | Train imgs: 12936 | Query: 3368 | Gallery: 15913
Batch shape: torch.Size([64, 3, 256, 128]) | pids sample: [350, 350, 350, 350, 737, 737, 737, 737]


## ধাপ ৪ — Model: SAM, CAM, STCA block, STCANet

**এটাই paper-এর মূল contribution।** তিনটা অংশ:

### (a) SAM — Spatial Attention Module
Paper-এর STAM-এর image-version (video না থাকায় temporal অংশ লাগে না, T=1)।
- feature map-কে ৪টা horizontal stripe-এ ভাগ করি (মাথা/উপর/নিচ/জুতা অনুমান — **সরলীকরণ note**: paper আসল PSPNet human-parsing ব্যবহার করেছিল, আমরা তার বদলে PCB-স্টাইল স্ট্রাইপ ব্যবহার করছি)
- প্রতিটা part-এর মধ্যে সম্পর্ক (relation) হিসাব করি — **Interconnection** (paper Eq. 2)
- সেই relation দিয়ে part feature-গুলো একত্র করি — **Accumulation** (Eq. 6)
- শেষে সব মিলিয়ে original feature map-এ ফিরিয়ে দেই (gate/residual আকারে) — **Transformation** (Eq. 7-8)

### (b) CAM — Channel Attention Module
feature map-এর প্রতিটা **channel**-এর মধ্যে semantic সম্পর্ক শেখে (Eq. 9-10) — ছোট ছোট distinguishing feature (যেমন জুতার রঙ) হারিয়ে না যায় তার জন্য।

### (c) STCA block = SAM + CAM + BatchNorm + residual (Eq. 11)
এই block-টা ResNet50-এর **layer2 এবং layer3-এর পরে** বসানো হয় (paper অনুযায়ী)।


In [25]:
import torch.nn as nn
import torch.nn.functional as F
import torchvision


class SAM(nn.Module):
    """Spatial Attention Module (STAM এর image-version, T=1)."""
    def __init__(self, in_channels, num_parts=4, reduction=8):
        super().__init__()
        self.P = num_parts
        self.C = in_channels
        hidden = max(in_channels // reduction, 32)

        self.relation_mlp = nn.Sequential(
            nn.Linear(in_channels * 2, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, 1),
        )
        self.part_transform = nn.Sequential(
            nn.Linear(in_channels * 2, in_channels),
            nn.ReLU(inplace=True),
        )
        self.global_transform = nn.Sequential(
            nn.Linear(in_channels * 2, in_channels),
            nn.ReLU(inplace=True),
        )
        self.out_proj = nn.Conv2d(in_channels, in_channels, kernel_size=1)

    def _split_parts(self, feat):
        B, C, H, W = feat.shape
        stripe_h = H // self.P
        parts = []
        for i in range(self.P):
            start = i * stripe_h
            end = (i + 1) * stripe_h if i < self.P - 1 else H
            stripe = feat[:, :, start:end, :]
            parts.append(F.adaptive_avg_pool2d(stripe, 1).view(B, C))
        return torch.stack(parts, dim=1)  # (B,P,C)

    def forward(self, feat):
        B, C, H, W = feat.shape
        PF = self._split_parts(feat)
        global_feat = F.adaptive_avg_pool2d(feat, 1).view(B, C)

        # ---- Interconnection: প্রতি part-pair এর মধ্যে relation (Eq. 2) ----
        PF_j = PF.unsqueeze(2).expand(-1, -1, self.P, -1)
        PF_k = PF.unsqueeze(1).expand(-1, self.P, -1, -1)
        diff = torch.abs(PF_j - PF_k)
        h_exp = global_feat.unsqueeze(1).unsqueeze(1).expand(-1, self.P, self.P, -1)
        rel_input = torch.cat([diff, h_exp], dim=-1)
        SR = self.relation_mlp(rel_input).squeeze(-1)

        eye = torch.eye(self.P, device=feat.device).bool()
        SR = SR.masked_fill(eye.unsqueeze(0), float("-inf"))
        SR = torch.softmax(SR, dim=-1)
        SR = torch.nan_to_num(SR)

        # ---- Accumulation (Eq. 6) ----
        A_STC = torch.bmm(SR, PF)

        # ---- Transformation (Eq. 7-8) ----
        fused_parts = torch.cat([PF, A_STC], dim=-1)
        PD_hat = self.part_transform(fused_parts)
        pooled = PD_hat.mean(dim=1)
        B_STS = self.global_transform(torch.cat([pooled, global_feat], dim=-1))

        gate = torch.sigmoid(self.out_proj(B_STS.view(B, C, 1, 1)))
        return feat * gate + feat


class CAM(nn.Module):
    """Channel Attention Module (Eq. 9-10)."""
    def __init__(self, in_channels):
        super().__init__()
        self.transform = nn.Conv2d(in_channels, in_channels, kernel_size=1)

    def forward(self, feat):
        B, C, H, W = feat.shape
        x = feat.view(B, C, H * W)
        energy = torch.bmm(x, x.transpose(1, 2))
        CR = torch.softmax(energy, dim=-1)
        Ac = torch.bmm(CR, x).view(B, C, H, W)
        Bc = self.transform(Ac)
        return feat + Bc


class STCA(nn.Module):
    """SAM + CAM + BatchNorm + residual (Eq. 11)."""
    def __init__(self, in_channels, num_parts=4):
        super().__init__()
        self.sam = SAM(in_channels, num_parts=num_parts)
        self.cam = CAM(in_channels)
        self.bn = nn.BatchNorm2d(in_channels)

    def forward(self, feat):
        out = self.sam(feat)
        out = self.cam(out)
        return self.bn(out) + feat


class STCANet(nn.Module):
    """Modified ResNet50 + STCA blocks after layer2 & layer3."""
    def __init__(self, num_classes, num_parts=4, pretrained=True):
        super().__init__()
        resnet = torchvision.models.resnet50(
            weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        )
        # last spatial down-sampling সরিয়ে দিচ্ছি (stride 2->1), PCB trick, feature map বড় রাখে
        resnet.layer4[0].conv2.stride = (1, 1)
        resnet.layer4[0].downsample[0].stride = (1, 1)

        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.stca2 = STCA(in_channels=512, num_parts=num_parts)
        self.layer3 = resnet.layer3
        self.stca3 = STCA(in_channels=1024, num_parts=num_parts)
        self.layer4 = resnet.layer4

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.bn_neck = nn.BatchNorm1d(2048)
        self.bn_neck.bias.requires_grad_(False)   # BNNeck trick, training stabilize করে
        self.classifier = nn.Linear(2048, num_classes, bias=False)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.stca2(x)
        x = self.layer3(x)
        x = self.stca3(x)
        x = self.layer4(x)

        feat = self.gap(x).flatten(1)
        bn_feat = self.bn_neck(feat)

        if self.training:
            logits = self.classifier(bn_feat)
            return logits, feat
        return bn_feat   # test-এ শুধু retrieval feature লাগবে

print("Model classes defined OK")


Model classes defined OK


### Sanity check — মডেল ঠিকমতো forward pass করছে কিনা টেস্ট করি (dummy data দিয়ে, dataset ছাড়াই)

In [26]:
_test_model = STCANet(num_classes=100, num_parts=4, pretrained=False)
_test_model.train()
_x = torch.randn(4, 3, 256, 128)
_logits, _emb = _test_model(_x)
print("logits:", _logits.shape, "| embedding:", _emb.shape)
del _test_model, _x, _logits, _emb
print("Model forward pass OK ✅")


logits: torch.Size([4, 100]) | embedding: torch.Size([4, 2048])
Model forward pass OK ✅


## ধাপ ৫ — Loss Function

Paper Section 4.3 (Eq. 15-17):
- **ID loss**: cross-entropy, person identity classification-এর জন্য
- **Triplet loss**: একই identity-র feature কাছাকাছি, ভিন্ন identity-র feature দূরে রাখার জন্য (batch-hard mining, margin=0.3)

`L_total = L_id + β1 * L_triplet`


In [28]:
class TripletLoss(nn.Module):
    """Batch-hard triplet loss, cosine distance (Eq. 16)."""
    def __init__(self, margin=0.3):
        super().__init__()
        self.ranking_loss = nn.MarginRankingLoss(margin=margin)

    def forward(self, embeddings, labels):
        embeddings = F.normalize(embeddings, dim=1)
        dist_mat = 1 - torch.mm(embeddings, embeddings.t())

        n = embeddings.size(0)
        labels = labels.view(-1, 1)
        same_id = labels.eq(labels.t())

        dist_ap, dist_an = [], []
        for i in range(n):
            pos_dist = dist_mat[i][same_id[i]]
            neg_dist = dist_mat[i][~same_id[i]]
            if pos_dist.numel() == 0 or neg_dist.numel() == 0:
                continue
            dist_ap.append(pos_dist.max().unsqueeze(0))
            dist_an.append(neg_dist.min().unsqueeze(0))

        dist_ap = torch.cat(dist_ap)
        dist_an = torch.cat(dist_an)
        y = torch.ones_like(dist_an)
        return self.ranking_loss(dist_an, dist_ap, y)


class STCANetLoss(nn.Module):
    def __init__(self, num_classes, margin=0.3, beta1=1.0, label_smoothing=0.1):
        super().__init__()
        self.id_loss = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
        self.triplet_loss = TripletLoss(margin=margin)
        self.beta1 = beta1

    def forward(self, logits, embeddings, labels):
        l_id = self.id_loss(logits, labels)
        l_tri = self.triplet_loss(embeddings, labels)
        total = l_id + self.beta1 * l_tri
        return total, l_id.item(), l_tri.item()

print("Loss classes defined OK")


Loss classes defined OK


## ধাপ ৬ — Evaluation Metrics (mAP + CMC)

**mAP (mean Average Precision, Eq. 13)**: প্রতিটা query-র জন্য গ্যালারি থেকে সঠিক match কতটা ভালোভাবে খুঁজে পেল তার গড়।

**CMC Rank-k (Eq. 14)**: top-k এর মধ্যে সঠিক identity পাওয়া গেলে ১, নাহলে ০ — সব query-র উপর গড় করে rank-1, rank-5 ইত্যাদি accuracy বের হয়।

Market-1501 প্রোটোকল অনুযায়ী: একই ক্যামেরা + একই identity-র গ্যালারি ছবি বাদ দেওয়া হয় (কারণ সেটা "trivial match", আসল re-identification না)।


In [29]:
import numpy as np


@torch.no_grad()
def extract_features(model, data_loader, device):
    model.eval()
    feats, pids, camids = [], [], []
    for imgs, pid, camid, _ in data_loader:
        imgs = imgs.to(device)
        f = model(imgs)
        f = F.normalize(f, dim=1)
        feats.append(f.cpu())
        pids.extend(pid.numpy())
        camids.extend(camid.numpy())
    return torch.cat(feats, dim=0), np.array(pids), np.array(camids)


def compute_mAP_CMC(dist_mat, q_pids, g_pids, q_camids, g_camids, max_rank=20):
    num_q, num_g = dist_mat.shape
    indices = np.argsort(dist_mat, axis=1)
    matches = (g_pids[indices] == q_pids[:, np.newaxis]).astype(np.int32)

    all_cmc, all_AP = [], []
    num_valid_q = 0.

    for q_idx in range(num_q):
        q_pid, q_camid = q_pids[q_idx], q_camids[q_idx]
        order = indices[q_idx]
        remove = (g_pids[order] == q_pid) & (g_camids[order] == q_camid)
        keep = np.invert(remove)

        raw_cmc = matches[q_idx][keep]
        if not np.any(raw_cmc):
            continue

        cmc = raw_cmc.cumsum()
        cmc[cmc > 1] = 1
        cmc = cmc[:max_rank]
        if cmc.shape[0] < max_rank:
            pad = np.full(max_rank - cmc.shape[0], cmc[-1] if cmc.shape[0] > 0 else 0)
            cmc = np.concatenate([cmc, pad])
        all_cmc.append(cmc)
        num_valid_q += 1.

        num_rel = raw_cmc.sum()
        tmp_cmc = raw_cmc.cumsum()
        tmp_cmc = [x / (i + 1.) for i, x in enumerate(tmp_cmc)]
        tmp_cmc = np.asarray(tmp_cmc) * raw_cmc
        AP = tmp_cmc.sum() / num_rel
        all_AP.append(AP)

    assert num_valid_q > 0, "No valid query found."
    all_cmc = np.asarray(all_cmc).astype(np.float32).sum(0) / num_valid_q
    mAP = np.mean(all_AP)
    return mAP, all_cmc

print("Eval functions defined OK")


Eval functions defined OK


## ধাপ ৭ — Re-ranking (k-reciprocal)

Paper Section 3.7: প্রাথমিক distance দিয়ে যে ranking পাওয়া যায়, সেটাকে আরও ভালো করার জন্য **k-reciprocal re-ranking** ব্যবহার করা হয় (Zhong et al. 2017, paper reference [39])। এটা একটা standard, widely-used algorithm — paper নিজেও এটা from-scratch বানায়নি, existing method ব্যবহার করেছে।

সংক্ষেপে: শুধু query-gallery distance না দেখে, gallery-এর মধ্যেও কারা কারা একে অপরের "reciprocal neighbor" সেটা দেখে re-rank করে — এতে ambiguous case-এ accuracy বাড়ে।


In [30]:
def re_ranking(q_feats, g_feats, k1=20, k2=6, lambda_value=0.3):
    q_feats = q_feats.cpu().numpy()
    g_feats = g_feats.cpu().numpy()
    all_feats = np.concatenate([q_feats, g_feats], axis=0)
    query_num, all_num = q_feats.shape[0], all_feats.shape[0]

    original_dist = 2 - 2 * np.dot(all_feats, all_feats.T)
    original_dist = np.power(original_dist, 2).astype(np.float32)
    original_dist = np.transpose(1. * original_dist / np.max(original_dist, axis=0))

    V = np.zeros_like(original_dist).astype(np.float32)
    initial_rank = np.argsort(original_dist).astype(np.int32)

    for i in range(all_num):
        forward_k_neigh_index = initial_rank[i, :k1 + 1]
        backward_k_neigh_index = initial_rank[forward_k_neigh_index, :k1 + 1]
        fi = np.where(backward_k_neigh_index == i)[0]
        k_reciprocal_index = forward_k_neigh_index[fi]
        k_reciprocal_expansion_index = k_reciprocal_index

        for j in range(len(k_reciprocal_index)):
            candidate = k_reciprocal_index[j]
            candidate_forward_k_neigh_index = initial_rank[candidate, :int(np.around(k1 / 2)) + 1]
            candidate_backward_k_neigh_index = initial_rank[candidate_forward_k_neigh_index,
                                                              :int(np.around(k1 / 2)) + 1]
            fi_candidate = np.where(candidate_backward_k_neigh_index == candidate)[0]
            candidate_k_reciprocal_index = candidate_forward_k_neigh_index[fi_candidate]
            if len(np.intersect1d(candidate_k_reciprocal_index, k_reciprocal_index)) > 2. / 3 * len(
                    candidate_k_reciprocal_index):
                k_reciprocal_expansion_index = np.append(k_reciprocal_expansion_index,
                                                          candidate_k_reciprocal_index)

        k_reciprocal_expansion_index = np.unique(k_reciprocal_expansion_index)
        weight = np.exp(-original_dist[i, k_reciprocal_expansion_index])
        V[i, k_reciprocal_expansion_index] = weight / np.sum(weight)

    original_dist = original_dist[:query_num, ]
    if k2 != 1:
        V_qe = np.zeros_like(V, dtype=np.float32)
        for i in range(all_num):
            V_qe[i, :] = np.mean(V[initial_rank[i, :k2], :], axis=0)
        V = V_qe

    invIndex = [np.where(V[:, i] != 0)[0] for i in range(all_num)]
    jaccard_dist = np.zeros_like(original_dist, dtype=np.float32)

    for i in range(query_num):
        temp_min = np.zeros(shape=[1, all_num], dtype=np.float32)
        indNonZero = np.where(V[i, :] != 0)[0]
        indImages = [invIndex[ind] for ind in indNonZero]
        for j in range(len(indNonZero)):
            temp_min[0, indImages[j]] = temp_min[0, indImages[j]] + \
                np.minimum(V[i, indNonZero[j]], V[indImages[j], indNonZero[j]])
        jaccard_dist[i] = 1 - temp_min / (2. - temp_min)

    final_dist = jaccard_dist * (1 - lambda_value) + original_dist * lambda_value
    final_dist = final_dist[:query_num, query_num:]
    return final_dist

print("Re-ranking function defined OK")


Re-ranking function defined OK


## ধাপ ৮ — Training Loop

এখন সবকিছু জোড়া দিয়ে training চালাব।

**সময়ের হিসাব (গুরুত্বপূর্ণ):** Paper-এ 60 epoch লাগানো হয়েছিল, ~6 ঘণ্টা GTX 3060Ti-তে। Kaggle-এর T4/P100-এ প্রতি epoch-এর সময় দেখে (নিচের log থেকে) হিসাব করে নাও কত epoch বাস্তবসম্মত।

Checkpoint প্রতি ৫ epoch-এ সেভ হবে (`/kaggle/working/checkpoints/`) — session টাইমআউট হলেও কাজ হারাবে না।


In [31]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training on:", device)

model = STCANet(num_classes=num_train_pids, num_parts=cfg.num_parts, pretrained=True).to(device)
criterion = STCANetLoss(num_train_pids, margin=cfg.triplet_margin, beta1=cfg.beta1)
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=cfg.lr_decay_epochs,
                                             gamma=cfg.lr_decay_factor)


Training on: cuda


In [32]:
import time

def evaluate(model, query_loader, gallery_loader, device, use_rerank=False):
    q_feats, q_pids, q_camids = extract_features(model, query_loader, device)
    g_feats, g_pids, g_camids = extract_features(model, gallery_loader, device)

    if use_rerank:
        dist_mat = re_ranking(q_feats, g_feats, k1=20, k2=6, lambda_value=0.3)
    else:
        dist_mat = (1 - torch.mm(q_feats, g_feats.t())).numpy()

    mAP, cmc = compute_mAP_CMC(dist_mat, q_pids, g_pids, q_camids, g_camids)
    print(f"mAP: {mAP*100:.1f}  Rank-1: {cmc[0]*100:.1f}  Rank-5: {cmc[4]*100:.1f}  "
          f"Rank-10: {cmc[9]*100:.1f}  Rank-20: {cmc[19]*100:.1f}")
    return mAP, cmc


for epoch in range(1, cfg.num_epochs + 1):
    model.train()
    t0 = time.time()
    running_loss, running_id, running_tri = 0., 0., 0.

    for it, (imgs, pids, camids, _) in enumerate(train_loader):
        imgs, pids = imgs.to(device), pids.to(device)

        logits, embeddings = model(imgs)
        loss, l_id, l_tri = criterion(logits, embeddings, pids)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_id += l_id
        running_tri += l_tri

        if (it + 1) % cfg.log_interval == 0:
            print(f"Epoch [{epoch}/{cfg.num_epochs}] Iter [{it+1}/{len(train_loader)}] "
                  f"Loss: {running_loss/(it+1):.4f} (ID: {running_id/(it+1):.4f}, "
                  f"Tri: {running_tri/(it+1):.4f})")

    scheduler.step()
    print(f"Epoch {epoch} done in {time.time()-t0:.1f}s | avg loss {running_loss/len(train_loader):.4f}")

    if epoch % 5 == 0 or epoch == cfg.num_epochs:
        ckpt_path = os.path.join(cfg.checkpoint_dir, f"stcanet_epoch{epoch}.pth")
        torch.save(model.state_dict(), ckpt_path)
        print(f"Saved checkpoint: {ckpt_path}")
        evaluate(model, query_loader, gallery_loader, device, use_rerank=False)


Epoch [1/50] Iter [50/46] Loss: 7.0529 (ID: 6.6337, Tri: 0.4192)
Epoch [1/50] Iter [100/46] Loss: 6.9018 (ID: 6.4790, Tri: 0.4228)
Epoch [1/50] Iter [150/46] Loss: 6.5800 (ID: 6.1649, Tri: 0.4151)
Epoch 1 done in 136.0s | avg loss 24.7950
Epoch [2/50] Iter [50/46] Loss: 5.6552 (ID: 5.3059, Tri: 0.3493)
Epoch [2/50] Iter [100/46] Loss: 5.1792 (ID: 4.8459, Tri: 0.3332)
Epoch [2/50] Iter [150/46] Loss: 4.6779 (ID: 4.3589, Tri: 0.3190)
Epoch 2 done in 136.9s | avg loss 17.2141
Epoch [3/50] Iter [50/46] Loss: 3.6832 (ID: 3.4122, Tri: 0.2710)
Epoch [3/50] Iter [100/46] Loss: 3.3541 (ID: 3.0868, Tri: 0.2672)
Epoch [3/50] Iter [150/46] Loss: 3.0645 (ID: 2.8037, Tri: 0.2608)
Epoch 3 done in 137.0s | avg loss 11.5054
Epoch [4/50] Iter [50/46] Loss: 2.5415 (ID: 2.2989, Tri: 0.2426)
Epoch [4/50] Iter [100/46] Loss: 2.3732 (ID: 2.1371, Tri: 0.2361)
Epoch [4/50] Iter [150/46] Loss: 2.2501 (ID: 2.0178, Tri: 0.2323)
Epoch 4 done in 136.4s | avg loss 8.5924
Epoch [5/50] Iter [50/46] Loss: 2.0171 (ID: 1

## ধাপ ৯ — Final Evaluation (with Re-ranking)

Training শেষে re-ranking সহ final score বের করি — এটাই paper-এর Table 3-এর সাথে তুলনীয় নম্বর (target: mAP 94.5 / Rank-1 95.5, তবে সরলীকরণের কারণে তোমার সংখ্যা কাছাকাছি হবে, হুবহু না)।


In [33]:
print("=== Final evaluation (without re-ranking) ===")
evaluate(model, query_loader, gallery_loader, device, use_rerank=False)

print("\n=== Final evaluation (with re-ranking) ===")
evaluate(model, query_loader, gallery_loader, device, use_rerank=True)


=== Final evaluation (without re-ranking) ===
mAP: 66.0  Rank-1: 85.4  Rank-5: 94.3  Rank-10: 96.9  Rank-20: 98.2

=== Final evaluation (with re-ranking) ===
mAP: 83.9  Rank-1: 88.8  Rank-5: 93.6  Rank-10: 95.6  Rank-20: 97.2


(np.float64(0.8393528153088093),
 array([0.88836104, 0.91270787, 0.9236936 , 0.93052256, 0.9361639 ,
        0.9409145 , 0.9453682 , 0.94922805, 0.9524941 , 0.956057  ,
        0.9584323 , 0.9602138 , 0.96199524, 0.9631829 , 0.9655582 ,
        0.9673397 , 0.9682304 , 0.96941805, 0.9706057 , 0.9723872 ],
       dtype=float32))

## পরবর্তী ধাপ / Troubleshooting

- **CUDA out of memory** → `cfg.batch_size_train` কমাও (64 → 32 বা 16)
- **Training ধীর** → epoch সংখ্যা কমিয়ে প্রথমে সব ঠিক আছে verify করো, তারপর বাড়াও
- **Session timeout হলে** → checkpoint (`/kaggle/working/checkpoints/*.pth`) থেকে `model.load_state_dict(torch.load(path))` দিয়ে load করে আবার চালাতে পারবে (নতুন cell-এ)
- **Loss NaN হলে** → learning rate কমাও, অথবা `torch.nan_to_num` জায়গাগুলো চেক করো

এই নম্বরগুলো (mAP, Rank-1/5/10/20) তোমার রিপোর্টে paper-এর Table 3-এর সাথে পাশাপাশি বসিয়ে তুলনা করবে, আর সরলীকরণ (PSPNet-এর বদলে stripe-based part, কম epoch) সততার সাথে উল্লেখ করবে।
